# Getting Started with FerroML

FerroML is a statistically rigorous machine learning library written in Rust with Python bindings. Unlike traditional ML libraries that focus only on predictions, FerroML provides **full statistical diagnostics** with every model:

- Coefficient confidence intervals and p-values
- R-style regression summaries
- Out-of-bag error estimation
- Feature importance with confidence intervals
- Cross-validation with proper statistical inference

This notebook will guide you through the basics of using FerroML.

## Installation

Install FerroML from PyPI:

```bash
pip install ferroml
```

Or build from source:

```bash
cd ferroml-python
pip install maturin
maturin develop --release
```

## 1. Linear Regression with Statistical Diagnostics

FerroML's linear regression provides R-style output with full statistical inference.

In [ ]:
import numpy as np
from ferroml.linear import LinearRegression

# Generate sample data
np.random.seed(42)
n_samples = 100
X = np.random.randn(n_samples, 3)
# True coefficients: [2, -1, 0.5]
y = 2 * X[:, 0] - 1 * X[:, 1] + 0.5 * X[:, 2] + np.random.randn(n_samples) * 0.5

# Fit the model
model = LinearRegression()
model.fit(X, y)

# Get R-style summary with full diagnostics
print(model.summary())

### Extracting Specific Statistics

You can also access individual statistics programmatically:

In [ ]:
# Model fit statistics
print(f"R-squared: {model.r_squared():.4f}")
print(f"Adjusted R-squared: {model.adjusted_r_squared():.4f}")
print(f"F-statistic: {model.f_statistic():.4f}")
print(f"F-statistic p-value: {model.f_pvalue():.6f}")

# Coefficients
print(f"\nCoefficients: {model.coef_}")
print(f"Intercept: {model.intercept_}")

### Coefficient Confidence Intervals

FerroML provides confidence intervals for all coefficients:

In [ ]:
# Get coefficient confidence intervals
ci = model.coefficients_with_ci(confidence_level=0.95)

print("Coefficient Analysis:")
print("-" * 60)
for i, coef_info in enumerate(ci):
    print(f"Feature {i}:")
    print(f"  Estimate: {coef_info['estimate']:.4f}")
    print(f"  Std Error: {coef_info['std_error']:.4f}")
    print(f"  t-statistic: {coef_info['t_statistic']:.4f}")
    print(f"  p-value: {coef_info['p_value']:.6f}")
    print(f"  95% CI: [{coef_info['ci_lower']:.4f}, {coef_info['ci_upper']:.4f}]")
    print()

### Prediction Intervals

Unlike sklearn, FerroML can provide prediction intervals:

In [ ]:
# Make predictions with intervals
X_new = np.array([[1.0, 0.5, -0.5], [0.0, 1.0, 1.0]])

predictions = model.predict(X_new)
intervals = model.predict_interval(X_new, confidence_level=0.95)

print("Predictions with 95% Prediction Intervals:")
for i, (pred, interval) in enumerate(zip(predictions, intervals)):
    print(f"Sample {i}: {pred:.4f} [{interval[0]:.4f}, {interval[1]:.4f}]")

## 2. Logistic Regression with Odds Ratios

FerroML's logistic regression includes odds ratios with confidence intervals:

In [ ]:
from ferroml.linear import LogisticRegression

# Generate binary classification data
np.random.seed(42)
n_samples = 200
X = np.random.randn(n_samples, 2)
y = (X[:, 0] + 0.5 * X[:, 1] + np.random.randn(n_samples) * 0.5 > 0).astype(np.float64)

# Fit logistic regression
lr = LogisticRegression()
lr.fit(X, y)

# Get summary
print(lr.summary())

In [ ]:
# Get odds ratios with confidence intervals
odds = lr.odds_ratios(confidence_level=0.95)

print("\nOdds Ratios with 95% CI:")
print("-" * 50)
for i, ratio in enumerate(odds):
    print(f"Feature {i}: OR = {ratio['odds_ratio']:.4f} [{ratio['ci_lower']:.4f}, {ratio['ci_upper']:.4f}]")

In [ ]:
# Probability predictions
X_test = np.array([[1.0, 0.5], [-1.0, -0.5]])
probabilities = lr.predict_proba(X_test)
predictions = lr.predict(X_test)

print("\nPredictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Sample {i}: Class {int(pred)}, P(class=1) = {prob:.4f}")

## 3. Tree-Based Models with OOB and Feature Importance

FerroML's Random Forest provides out-of-bag error estimation and feature importance with confidence intervals.

In [ ]:
from ferroml.trees import RandomForestClassifier

# Generate classification data
np.random.seed(42)
n_samples = 500
n_features = 5
X = np.random.randn(n_samples, n_features)
# Only first 2 features are informative
y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(np.float64)

# Fit Random Forest with OOB
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    oob_score=True,
    random_state=42
)
rf.fit(X, y)

print(f"OOB Score: {rf.oob_score_:.4f}")
print(f"\nFeature Importances:")
for i, imp in enumerate(rf.feature_importances_):
    print(f"  Feature {i}: {imp:.4f}")

### Gradient Boosting with Early Stopping

FerroML includes Histogram-based Gradient Boosting (similar to LightGBM):

In [ ]:
from ferroml.trees import HistGradientBoostingClassifier

# Split data for early stopping
train_size = int(0.8 * n_samples)
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

# Fit with early stopping
hgb = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_depth=5,
    early_stopping=True,
    n_iter_no_change=10,
    random_state=42
)
hgb.fit(X_train, y_train)

# Evaluate
train_pred = hgb.predict(X_train)
val_pred = hgb.predict(X_val)
train_acc = (train_pred == y_train).mean()
val_acc = (val_pred == y_val).mean()

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

## 4. Preprocessing

FerroML provides sklearn-compatible preprocessing transformers:

In [ ]:
from ferroml.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Generate sample data
np.random.seed(42)
X = np.random.randn(100, 3) * [1, 10, 100]  # Different scales
X[:5, 0] = 1000  # Add some outliers

print("Original data:")
print(f"  Mean: {X.mean(axis=0)}")
print(f"  Std: {X.std(axis=0)}")
print(f"  Min: {X.min(axis=0)}")
print(f"  Max: {X.max(axis=0)}")

In [ ]:
# StandardScaler: mean=0, std=1
scaler = StandardScaler()
X_standard = scaler.fit_transform(X)

print("StandardScaler:")
print(f"  Mean: {X_standard.mean(axis=0)}")
print(f"  Std: {X_standard.std(axis=0)}")

In [ ]:
# RobustScaler: robust to outliers
robust = RobustScaler()
X_robust = robust.fit_transform(X)

print("RobustScaler (better for data with outliers):")
print(f"  Median: {np.median(X_robust, axis=0)}")
print(f"  IQR: {np.percentile(X_robust, 75, axis=0) - np.percentile(X_robust, 25, axis=0)}")

### Handling Missing Values

In [ ]:
from ferroml.preprocessing import SimpleImputer

# Create data with missing values
X_missing = np.array([
    [1.0, 2.0, 3.0],
    [4.0, np.nan, 6.0],
    [7.0, 8.0, np.nan],
    [np.nan, 5.0, 9.0],
])

print("Data with missing values:")
print(X_missing)

# Impute with mean
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_missing)

print("\nAfter mean imputation:")
print(X_imputed)

## 5. Building Pipelines

FerroML supports sklearn-style pipelines:

In [ ]:
from ferroml.pipeline import Pipeline
from ferroml.preprocessing import StandardScaler
from ferroml.linear import LinearRegression

# Create a pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])

# Generate data
np.random.seed(42)
X = np.random.randn(100, 3)
y = 2 * X[:, 0] - X[:, 1] + 0.5 * X[:, 2] + np.random.randn(100) * 0.5

# Fit the pipeline
pipe.fit(X, y)

# Make predictions
predictions = pipe.predict(X[:5])
print("Pipeline predictions:")
for i, (pred, actual) in enumerate(zip(predictions, y[:5])):
    print(f"  Sample {i}: predicted={pred:.4f}, actual={actual:.4f}")

### Column Transformer for Mixed Data Types

In [ ]:
from ferroml.pipeline import ColumnTransformer
from ferroml.preprocessing import StandardScaler, OneHotEncoder

# Create data with numeric and categorical columns
# Columns 0, 1 are numeric; columns 2, 3 are categorical (as integers)
X_mixed = np.array([
    [1.0, 2.0, 0.0, 1.0],
    [3.0, 4.0, 1.0, 0.0],
    [5.0, 6.0, 2.0, 1.0],
    [7.0, 8.0, 0.0, 0.0],
])

# Apply different transformers to different columns
ct = ColumnTransformer([
    ('numeric', StandardScaler(), [0, 1]),
    ('categorical', OneHotEncoder(), [2, 3]),
])

X_transformed = ct.fit_transform(X_mixed)
print("Transformed shape:", X_transformed.shape)
print("Transformed data:")
print(X_transformed)

## 6. Regularized Regression

FerroML provides Ridge, Lasso, and Elastic Net with statistical diagnostics:

In [ ]:
from ferroml.linear import RidgeRegression, LassoRegression, ElasticNet

# Generate data with correlated features
np.random.seed(42)
n_samples = 100
X = np.random.randn(n_samples, 10)
# Add correlation
X[:, 5] = X[:, 0] + np.random.randn(n_samples) * 0.1
X[:, 6] = X[:, 1] + np.random.randn(n_samples) * 0.1
# Only first 3 features are truly informative
y = X[:, 0] - 0.5 * X[:, 1] + 2 * X[:, 2] + np.random.randn(n_samples) * 0.5

# Compare models
models = {
    'OLS': LinearRegression(),
    'Ridge': RidgeRegression(alpha=1.0),
    'Lasso': LassoRegression(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
}

print("Coefficient comparison (true: [1, -0.5, 2, 0, 0, 0, 0, 0, 0, 0]):")
print("-" * 80)
for name, model in models.items():
    model.fit(X, y)
    coef_str = ', '.join([f'{c:.3f}' for c in model.coef_])
    print(f"{name:12} [{coef_str}]")

Notice how:
- **OLS** spreads coefficients across correlated features
- **Ridge** shrinks coefficients but keeps all non-zero
- **Lasso** produces sparse solutions (zeros out uninformative features)
- **Elastic Net** combines L1 and L2 regularization

## 7. Quick Introduction to AutoML

FerroML's AutoML automatically searches for the best model with statistical rigor:

In [ ]:
from ferroml.automl import AutoML, AutoMLConfig

# Generate classification data
np.random.seed(42)
n_samples = 500
X = np.random.randn(n_samples, 5)
y = (X[:, 0] + 0.5 * X[:, 1] - 0.3 * X[:, 2] > 0).astype(np.float64)

# Configure AutoML
config = AutoMLConfig(
    task="classification",
    metric="accuracy",
    time_budget_seconds=60,  # 1 minute budget
    cv_folds=5,
)

# Run AutoML
automl = AutoML(config)
result = automl.fit(X, y)

# View results
print(result.summary())

In [ ]:
# Get the best model
best = result.best_model()
print(f"\nBest model: {best.algorithm}")
print(f"CV Score: {best.cv_score:.4f}")

# Get leaderboard
print("\nLeaderboard (top 5):")
for i, entry in enumerate(result.leaderboard()[:5]):
    print(f"  {i+1}. {entry.algorithm}: {entry.cv_score:.4f} (CI: [{entry.ci_lower:.4f}, {entry.ci_upper:.4f}])")

## Summary: FerroML vs sklearn

| Feature | sklearn | FerroML |
|---------|---------|----------|
| Basic fit/predict | Yes | Yes |
| Coefficient p-values | No | Yes |
| Coefficient CIs | No | Yes |
| Prediction intervals | No | Yes |
| Odds ratios | No | Yes |
| OOB with CI | Partial | Yes |
| Statistical model summaries | No | Yes |
| AutoML with significance tests | No | Yes |

FerroML is designed for practitioners who need **rigorous statistical inference**, not just predictions.

## Next Steps

- **02_statistical_diagnostics.ipynb**: Deep dive into statistical diagnostics
- **03_ferroml_vs_sklearn.ipynb**: Detailed comparison with sklearn
- **04_production_deployment.ipynb**: Export to ONNX and production deployment

For more information, see the [FerroML documentation](https://github.com/ferroml/ferroml).